# Korean Speech-Level Shift Tracking: Final Experiments

This notebook evaluates Qwen2.5-3B-Instruct on 224 synthetic
six-turn Korean exchanges constructed from eight scenario templates.

The task is to return NONE when no speaker changes speech level,
or the speaker and global turn number of the first within-speaker
change between Haeyo-che and Hae-che.

The six exploratory pilot scenarios are excluded from this evaluation.
The final dataset, prompts, and baseline rules were fixed before
examining the final model outputs.

The Korean examples were constructed with AI assistance and have not
received independent review by a proficient Korean speaker. Their
linguistic naturalness and proposed speech-level labels remain limitations.

## Execution

Cells 1–4 load the dataset and model and reproduce inference.
They require a GPU for model execution.

Cells 5–6 analyze saved JSON results without loading the model.
To reproduce the analysis only, run these two cells with:
- final_direct_results.json
- final_pipeline_results.json
- final_recognition_results.json

The dataset is supplied separately as final_dataset_v1.json.

In [ ]:
import json
import hashlib
from pathlib import Path
from google.colab import files

dataset_path = Path("final_dataset_v1.json")

if not dataset_path.exists():
    print("Sélectionne final_dataset_v1.json")
    uploaded = files.upload()
    matching = [
        name for name in uploaded
        if name.startswith("final_dataset_v1") and name.endswith(".json")
    ]
    if len(matching) != 1:
        raise ValueError("Sélectionne uniquement final_dataset_v1.json.")
    dataset_path.write_bytes(uploaded[matching[0]])

dataset_bytes = dataset_path.read_bytes()
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()

assert dataset_hash == (
    "bbf21bff6c99c2a059752d6fa6ef4d105"
    "2886e744944a552d0b15ccee8a41699"
), "Le fichier ne correspond pas au dataset figé."

final_data = json.loads(dataset_bytes)
final_items = final_data["items"]
SYSTEM_PROMPT = final_data["system_prompt"]

def track_first_shift(turns):
    last_level = {}
    for number, (speaker, level) in enumerate(turns, start=1):
        if speaker in last_level and last_level[speaker] != level:
            return f"({speaker}, {number})"
        last_level[speaker] = level
    return "NONE"

print("Dataset restauré :", len(final_items), "dialogues")
print(
    "Modèle en mémoire :",
    "model" in globals() and "tokenizer" in globals()
)

Sélectionne final_dataset_v1.json


Saving final_dataset_v1.json to final_dataset_v1.json
Dataset restauré : 224 dialogues
Modèle en mémoire : False


In [ ]:
%pip -q install transformers accelerate bitsandbytes

import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

set_seed(42)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

if "model" not in globals():
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16,
    )
    model.eval()

def predict_shift(dialogue):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": dialogue},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

assert torch.cuda.is_available(), "Active le GPU T4."
print("Modèle prêt :", MODEL_ID)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modèle prêt : Qwen/Qwen2.5-3B-Instruct


In [ ]:
import json
import hashlib
from pathlib import Path
import pandas as pd
from google.colab import files

dataset_path = Path("final_dataset_v1.json")
dataset_bytes = dataset_path.read_bytes()
final_data = json.loads(dataset_bytes)
dataset_hash = hashlib.sha256(dataset_bytes).hexdigest()

assert dataset_hash == (
    "bbf21bff6c99c2a059752d6fa6ef4d105"
    "2886e744944a552d0b15ccee8a41699"
)

SYSTEM_PROMPT = final_data["system_prompt"]
final_items = final_data["items"]

output_path = Path("final_direct_results.json")
run = {
    "dataset_sha256": dataset_hash,
    "model_id": MODEL_ID,
    "system_prompt": SYSTEM_PROMPT,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "seed": 42,
    "generation": {"do_sample": False, "max_new_tokens": 32},
    "results": [],
}

# Resume an interrupted run in the same Colab session.
if output_path.exists():
    run = json.loads(output_path.read_text(encoding="utf-8"))
    assert run["dataset_sha256"] == dataset_hash
    assert run["system_prompt"] == SYSTEM_PROMPT

completed = {row["id"] for row in run["results"]}

for item in final_items:
    if item["id"] in completed:
        continue

    dialogue = "\n".join(
        f'{t["turn"]}. {t["speaker"]}: {t["text"]}'
        for t in item["turns"]
    )
    prediction = predict_shift(dialogue)

    run["results"].append({
        **item,
        "dialogue": dialogue,
        "prediction": prediction,
        "correct": (
            "".join(prediction.split())
            == "".join(item["expected"].split())
        ),
    })

    output_path.write_text(
        json.dumps(run, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if len(run["results"]) % 20 == 0:
        print(f'{len(run["results"])}/{len(final_items)} terminés')

final_direct_df = pd.DataFrame(run["results"])

print("\nH1 — réponses correctes :")
print(
    final_direct_df.query("experiment == 'H1'")
    .groupby("condition")["correct"].agg(["sum", "count", "mean"])
)

print("\nH2 — localisation correcte par distance :")
print(
    final_direct_df.query("experiment == 'H2'")
    .groupby("gap")["correct"].agg(["sum", "count", "mean"])
)

print("\nH2 — localisation correcte par personne :")
print(
    final_direct_df.query("experiment == 'H2'")
    .groupby("target")["correct"].agg(["sum", "count", "mean"])
)

files.download(str(output_path))

20/224 terminés
40/224 terminés
60/224 terminés
80/224 terminés
100/224 terminés
120/224 terminés
140/224 terminés
160/224 terminés
180/224 terminés
200/224 terminés
220/224 terminés

H1 — réponses correctes :
                    sum  count   mean
condition                            
contrasting_levels   10     16  0.625
same_level           10     16  0.625

H2 — localisation correcte par distance :
     sum  count      mean
gap                      
0.0    0     64  0.000000
1.0    5     64  0.078125
2.0    4     64  0.062500

H2 — localisation correcte par personne :
        sum  count     mean
target                     
A         0     96  0.00000
B         9     96  0.09375


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re
import json
from pathlib import Path
import pandas as pd
from google.colab import files

# Frozen pilot rule: do not expand it after seeing test results.
def rule_level(text):
    cleaned = re.sub(r"[\s.!?。！？]+$", "", text)

    if cleaned.endswith("요"):
        return "HAEYO"
    if cleaned.endswith(("어", "아", "해", "와", "워", "싸", "셔")):
        return "HAE"
    return "UNKNOWN"


LEVEL_PROMPT = """Classify the addressee speech level of the Korean utterance.
Output exactly HAEYO for polite Haeyo-che or HAE for informal Hae-che.
Do not explain your answer."""

def predict_level(utterance):
    messages = [
        {"role": "system", "content": LEVEL_PROMPT},
        {"role": "user", "content": utterance},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=16,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()


# Classify each distinct sentence once; save after every prediction.
cache_path = Path("final_recognition_results.json")
recognition_run = {
    "dataset_sha256": dataset_hash,
    "prompt": LEVEL_PROMPT,
    "results": [],
}

if cache_path.exists():
    recognition_run = json.loads(cache_path.read_text(encoding="utf-8"))
    assert recognition_run["dataset_sha256"] == dataset_hash
    assert recognition_run["prompt"] == LEVEL_PROMPT

lookup = {
    row["text"]: row["prediction"]
    for row in recognition_run["results"]
}

sentences = {}
for item in final_items:
    for turn in item["turns"]:
        text, level = turn["text"], turn["level"]
        assert text not in sentences or sentences[text] == level
        sentences[text] = level

for text, expected in sorted(sentences.items()):
    if text in lookup:
        continue

    prediction = predict_level(text)
    lookup[text] = prediction

    recognition_run["results"].append({
        "text": text,
        "expected": expected,
        "prediction": prediction,
        "correct": prediction == expected,
    })
    cache_path.write_text(
        json.dumps(recognition_run, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

print("Reconnaissance des phrases distinctes :")
print(
    pd.DataFrame(recognition_run["results"])
    .groupby("expected")["correct"].agg(["sum", "count", "mean"])
)


# Evaluate both pipelines.
pipeline_results = []

for item in final_items:
    for method in ["rules_tracking", "model_tracking"]:
        levels = [
            (
                turn["speaker"],
                rule_level(turn["text"])
                if method == "rules_tracking"
                else lookup[turn["text"]],
            )
            for turn in item["turns"]
        ]

        prediction = (
            "UNCLASSIFIED"
            if any(level not in {"HAEYO", "HAE"} for _, level in levels)
            else track_first_shift(levels)
        )

        pipeline_results.append({
            **item,
            "method": method,
            "prediction": prediction,
            "correct": prediction == item["expected"],
        })

pipeline_df = pd.DataFrame(pipeline_results)

print("\nH1 — réponses correctes :")
print(
    pipeline_df.query("experiment == 'H1'")
    .groupby(["method", "condition"])["correct"]
    .agg(["sum", "count", "mean"])
)

print("\nH2 — localisation exacte :")
print(
    pipeline_df.query("experiment == 'H2'")
    .groupby(["method", "gap"])["correct"]
    .agg(["sum", "count", "mean"])
)

print("\nDialogues non classés :")
print(
    pipeline_df.assign(
        unclassified=pipeline_df["prediction"].eq("UNCLASSIFIED")
    ).groupby("method")["unclassified"].sum()
)

with open("final_pipeline_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "dataset_sha256": dataset_hash,
        "results": pipeline_results,
    }, f, ensure_ascii=False, indent=2)

files.download("final_recognition_results.json")
files.download("final_pipeline_results.json")

Reconnaissance des phrases distinctes :
          sum  count      mean
expected                      
HAE        25     48  0.520833
HAEYO      27     48  0.562500

H1 — réponses correctes :
                                   sum  count    mean
method         condition                             
model_tracking contrasting_levels    0     16  0.0000
               same_level            0     16  0.0000
rules_tracking contrasting_levels   11     16  0.6875
               same_level           11     16  0.6875

H2 — localisation exacte :
                    sum  count    mean
method         gap                    
model_tracking 0.0    0     64  0.0000
               1.0    0     64  0.0000
               2.0    0     64  0.0000
rules_tracking 0.0   44     64  0.6875
               1.0   44     64  0.6875
               2.0   44     64  0.6875

Dialogues non classés :
method
model_tracking     0
rules_tracking    70
Name: unclassified, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Quantitative analysis

We compare direct model prediction, rule-based recognition followed
by deterministic speaker tracking, and model-based recognition followed
by the same tracker. Unclassified outputs count as incorrect in accuracy.

H1 compares false-positive shift rates between contrasting-level and
same-level no-shift exchanges.

H2 compares exact speaker-and-turn localization across 0, 1, and 2
intervening turns. All exchanges contain six turns, and the true shift
occurs at turn 5. The preceding target utterance changes position,
so distance is not isolated from every positional effect.

Confidence intervals use 10,000 paired bootstrap resamples of the eight
scenario templates, with seed 42. Related variants remain grouped.
These intervals do not account for linguistic annotation uncertainty.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import files

required_files = [
    "final_direct_results.json",
    "final_pipeline_results.json",
    "final_recognition_results.json",
]

missing = [name for name in required_files if not Path(name).exists()]

if missing:
    print("Sélectionne ces fichiers :", ", ".join(missing))
    files.upload()

for name in required_files:
    if not Path(name).exists():
        raise FileNotFoundError(f"Fichier manquant : {name}")

direct = json.loads(Path(required_files[0]).read_text(encoding="utf-8"))
pipelines = json.loads(Path(required_files[1]).read_text(encoding="utf-8"))
recognition = json.loads(Path(required_files[2]).read_text(encoding="utf-8"))

assert (
    direct["dataset_sha256"]
    == pipelines["dataset_sha256"]
    == recognition["dataset_sha256"]
)

rows = [
    {**row, "method": "direct"}
    for row in direct["results"]
] + pipelines["results"]

df = pd.DataFrame(rows)

# Recompute correctness from predictions rather than trusting saved scores.
def normalize(text):
    return "".join(text.split())

df["correct"] = [
    normalize(pred) == normalize(expected)
    for pred, expected in zip(df["prediction"], df["expected"])
]

assert not df.duplicated(["method", "id"]).any()
assert df.groupby("method").size().eq(224).all()

print("TABLEAU PRINCIPAL — exactitude")
main_table = (
    df.groupby(["method", "experiment"])["correct"]
    .agg(["sum", "count", "mean"])
)
print(main_table)

print("\nH1 — exactitude par condition")
print(
    df[df["experiment"] == "H1"]
    .groupby(["method", "condition"])["correct"]
    .agg(["sum", "count", "mean"])
)

print("\nH2 — localisation exacte par distance")
print(
    df[df["experiment"] == "H2"]
    .groupby(["method", "gap"])["correct"]
    .agg(["sum", "count", "mean"])
)

# Paired bootstrap over scenarios, preserving all variants together.
# H1: contrasting-level minus same-level false-positive rate.
# UNCLASSIFIED is an abstention, not a false-positive shift.
# H2: accuracy at gap 2 minus accuracy at gap 0.
bootstrap_rows = []

for method in sorted(df["method"].unique()):
    subset = df[df["method"] == method].copy()

    h1 = subset[subset["experiment"] == "H1"].copy()
    h1["false_positive"] = h1["prediction"].apply(
        lambda value: re.fullmatch(
            r"\(\s*[AB]\s*,\s*\d+\s*\)", value.strip()
        ) is not None
    )

    h1_rates = h1.groupby(
        ["template_id", "condition"]
    )["false_positive"].mean().unstack()

    h2_rates = (
        subset[subset["experiment"] == "H2"]
        .groupby(["template_id", "gap"])["correct"]
        .mean().unstack()
    )

    contrasts = {
        "H1_FPR_contrasting_minus_same": (
            h1_rates["contrasting_levels"] - h1_rates["same_level"]
        ),
        "H2_accuracy_gap2_minus_gap0": (
            h2_rates[2] - h2_rates[0]
        ),
    }

    for contrast, values in contrasts.items():
        values = values.sort_index().to_numpy(dtype=float)
        rng = np.random.default_rng(42)

        samples = rng.choice(
            values,
            size=(10000, len(values)),
            replace=True,
        ).mean(axis=1)

        low, high = np.quantile(samples, [0.025, 0.975])

        bootstrap_rows.append({
            "method": method,
            "contrast": contrast,
            "estimate": values.mean(),
            "CI95_low": low,
            "CI95_high": high,
            "scenarios": len(values),
        })

bootstrap_df = pd.DataFrame(bootstrap_rows)

print("\nCONTRASTES — proportions et IC bootstrap à 95 %")
print(bootstrap_df.to_string(index=False))

main_table.to_csv("main_results.csv")
bootstrap_df.to_csv("bootstrap_contrasts.csv", index=False)

print("\nAnalyse terminée. Aucun appel au modèle.")

Sélectionne ces fichiers : final_direct_results.json, final_pipeline_results.json, final_recognition_results.json


Saving final_direct_results.json to final_direct_results.json
Saving final_pipeline_results.json to final_pipeline_results.json
Saving final_recognition_results.json to final_recognition_results.json
TABLEAU PRINCIPAL — exactitude
                           sum  count      mean
method         experiment                      
direct         H1           20     32  0.625000
               H2            9    192  0.046875
model_tracking H1            0     32  0.000000
               H2            0    192  0.000000
rules_tracking H1           22     32  0.687500
               H2          132    192  0.687500

H1 — exactitude par condition
                                   sum  count    mean
method         condition                             
direct         contrasting_levels   10     16  0.6250
               same_level           10     16  0.6250
model_tracking contrasting_levels    0     16  0.0000
               same_level            0     16  0.0000
rules_tracking contrasting_lev

In [2]:
# Requires df from the preceding analysis cell.
import re
import json
import random

def classify_error(row):
    prediction = row["prediction"].strip()

    if row["correct"]:
        return "correct"

    if prediction == "UNCLASSIFIED":
        return "abstention"

    if prediction == "NONE":
        return "missed_shift"

    match = re.fullmatch(r"\(\s*([AB])\s*,\s*(\d+)\s*\)", prediction)

    if match is None:
        return "malformed"

    speaker, turn = match.group(1), int(match.group(2))

    if not 1 <= turn <= len(row["turns"]):
        return "out_of_range"

    if row["turns"][turn - 1]["speaker"] != speaker:
        return "speaker_turn_mismatch"

    if row["expected"] == "NONE":
        return "false_positive"

    expected_match = re.fullmatch(
        r"\(\s*([AB])\s*,\s*(\d+)\s*\)", row["expected"]
    )
    expected_speaker = expected_match.group(1)
    expected_turn = int(expected_match.group(2))

    if speaker != expected_speaker and turn != expected_turn:
        return "wrong_speaker_and_turn"
    if speaker != expected_speaker:
        return "wrong_speaker"
    return "wrong_turn"


df["error_category"] = df.apply(classify_error, axis=1)

print("CATÉGORIES EXCLUSIVES — chaque réponse apparaît une seule fois")
error_table = pd.crosstab(
    [df["method"], df["experiment"]],
    df["error_category"],
)
print(error_table.to_string())

print("\nCOUVERTURE — proportion de dialogues sans abstention")
print(
    df.assign(classified=df["prediction"] != "UNCLASSIFIED")
    .groupby("method")["classified"].mean()
)

# Sample independently of the error categories.
# One random example per experiment and outcome for the direct model.
rng = random.Random(42)
sampled_examples = []

for experiment in ["H1", "H2"]:
    for correct in [True, False]:
        candidates = (
            df[
                (df["method"] == "direct")
                & (df["experiment"] == experiment)
                & (df["correct"] == correct)
            ]
            .sort_values("id")
            .to_dict("records")
        )

        if not candidates:
            continue

        row = rng.choice(candidates)
        example = {
            "id": row["id"],
            "experiment": experiment,
            "correct": correct,
            "expected": row["expected"],
            "prediction": row["prediction"],
            "error_category": row["error_category"],
            "turns": row["turns"],
        }
        sampled_examples.append(example)

        print(f'\nEXEMPLE : {example["id"]}')
        for turn in example["turns"]:
            print(f'{turn["turn"]}. {turn["speaker"]}: {turn["text"]}')
        print("Attendu :", example["expected"])
        print("Prédit :", example["prediction"])
        print("Catégorie :", example["error_category"])

error_table.to_csv("error_categories.csv")

with open("sampled_examples.json", "w", encoding="utf-8") as f:
    json.dump(sampled_examples, f, ensure_ascii=False, indent=2)

CATÉGORIES EXCLUSIVES — chaque réponse apparaît une seule fois
error_category             abstention  correct  false_positive  missed_shift  out_of_range  speaker_turn_mismatch  wrong_speaker_and_turn
method         experiment                                                                                                                
direct         H1                   0       20              11             0             0                      1                       0
               H2                   0        9               0            51             1                     10                     121
model_tracking H1                   0        0              32             0             0                      0                       0
               H2                   0        0               0             0             0                      0                     192
rules_tracking H1                  10       22               0             0             0                   

## Findings and limitations

The direct model achieves 62.5% accuracy on no-shift exchanges and
4.69% exact localization on shift exchanges. Its no-shift false-alarm
rate is 37.5% in both speech-level conditions, providing no support
for H1 in this evaluation.

Exact localization is 0%, 7.81%, and 6.25% at distances 0, 1, and 2.
This pattern does not support H2's predicted decline. Performance is
near the floor, limiting conclusions about distance sensitivity.

Isolated-utterance classification accuracy is 54.17%. Combining these
classifications with deterministic tracking produces no correct
dialogue-level answers in this dataset. This finding concerns the
specific model, prompt, and examples tested.

The frozen rule-based pipeline achieves 68.75% coverage and is correct
on every covered dialogue. Its 70 abstentions are counted as errors
in overall accuracy. This restricted baseline is not a general Korean
speech-level classifier.

The direct model also produces 11 speaker–turn inconsistencies and one
out-of-range turn number. Error categories are mutually exclusive:
structural inconsistencies are identified before localization errors.

Four qualitative examples are sampled with seed 42, stratified by
experiment and correctness. They illustrate outcomes but do not estimate
the prevalence of individual error mechanisms.

Conclusions are limited by eight synthetic scenarios, no independent
Korean-language validation, one quantized model, one main prompt,
restricted baseline coverage, and positional constraints. Zero-width
bootstrap intervals reflect identical observed scenario-level contrasts,
not proof that an effect is universally absent.